# 01 — Reprodução: detecção de mixers na Ethereum

Notebook didático de reprodução do projeto inspirado no artigo *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (SBSeg 2025).

**Escopo desta versão:** utilizar datasets processados e auditados, verificar integridade, validar e treinar Random Forest, e avaliar o dia 29/10/2020. **Não reprocessa dumps brutos** e não deve ser apresentado como validação cega. Código versionado no [GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/tree/main/src/ethmix); esta cópia é para execução no Colab.

**Unidade:** janela contrato-destinatário/dia UTC. Um endereço pode produzir várias janelas. Conjunto de treino legado tem 23 positivas de **um único pool** Tornado Cash 10 ETH em março/2025 e 100 negativas selecionadas por inspeção; isso limita a generalização.
**Dados oficiais:** Projeto_Mixers_Ethereum no seu Drive (não o atalho do Bruno).

**Acesso aos dados:** pelo ID de cada arquivo do seu Drive; não exige que a pasta apareça dentro de `MyDrive`.


## 1 — Conceitos e variáveis

| Feature | O que mede |
|---|---|
| `value_entropy` | Diversidade dos valores transferidos |
| `fee_entropy` | Diversidade das taxas |
| `internal_value_entropy` | Diversidade dos valores internos |
| `call_count_mean` | Média das chamadas da transação |

As entropias são Shannon base 2, depois de discretização por arredondamento (4 casas valores, 6 taxas). Parâmetros de engenharia deste projeto não são automaticamente parâmetros explícitos do artigo.

## 2 — Carregar o código diretamente do **seu** Google Drive

A pasta `Projeto_Mixers_Ethereum` está no Drive, mas pode não aparecer na montagem `/content/drive/MyDrive` (por exemplo, se estiver em um local compartilhado). **Não precisamos depender desse caminho.**

Esta célula solicita o login **Google** do Colab (não é token GitHub), acessa **somente os três módulos Python já copiados para o seu projeto**, pelos respectivos IDs do Drive, e confere SHA-256 para garantir que o código corresponde à versão registrada no GitHub.

Os módulos são baixados para a pasta temporária `/content/ethmix_reproducao/src/ethmix/`. Não alteramos dados da pasta compartilhada do Bruno.

In [ ]:
from pathlib import Path
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io, json, sys, hashlib

# Autoriza somente a conta Google no Colab. Não precisa montar o Meu Drive.
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_api = build('drive', 'v3', credentials=creds, cache_discovery=False)

REPO = 'BorgesGusta/ethereum-mixer-detection'
CODIGO_REFERENCIA = 'src/ethmix (GitHub, hash SHA-256 dos módulos)'
BASE = Path('/content/ethmix_reproducao')  # cache de execução, NÃO o Drive compartilhado

# IDs dos módulos copiados para o seu Projeto_Mixers_Ethereum/src/ethmix.
MODULOS_DRIVE = {
    '__init__.py': '1DnOyPTb_N_GztcgZ5BDp28vp5utfwBph',
    'features.py': '1rxO8fSvW5GABo7YmYVJtcuTTmMvnBQzL',
    'models.py': '10x_kaEy6pHRRYULoud5lFvFZLqGYy6OT',
}
EXPECTED_MODULE_SHA256 = {
    '__init__.py': 'dccb63bbe5fbde892d3059dab7e3efc03f2e0e169d93afa67e8d983e996cd1ae',
    'features.py': 'b75d65bd2326169e6fc52f86f5030337d6e6813df607bab0a9c3cb20922fd0b5',
    'models.py': '7f92d88cafcfc7c939f8c84e4fcfb34a0aa98c0890a86333c294b23f1508d6d9',
}

def obter_arquivo_drive(file_id, destino):
    """Baixa arquivo binário do próprio Drive por ID, sem precisar de um caminho montado."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    with destino.open('wb') as f:
        downloader = MediaIoBaseDownload(f, request, chunksize=4 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return destino

for nome, file_id in MODULOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, BASE / 'src' / 'ethmix' / nome)
    obtido = hashlib.sha256(arquivo.read_bytes()).hexdigest()
    if obtido != EXPECTED_MODULE_SHA256[nome]:
        raise ValueError(f'SHA-256 diferente para {nome}: {obtido}. Verifique a sincronização GitHub/Drive.')

sys.path.insert(0, str(BASE / 'src'))
# Evita módulos antigos da memória se uma versão anterior do notebook já foi executada.
for nome in ('ethmix.models', 'ethmix.features', 'ethmix'):
    sys.modules.pop(nome, None)
from ethmix.features import FEATURES, DECIMAIS, preparar, calcular_features
from ethmix.models import novo_rf, validar_cv, reamostrar
print('Código carregado do seu Drive (por ID), sem GitHub Token.')
print('Integridade SHA-256 dos três módulos: verificada.')

## 3 — Baixar os dados processados **da sua pasta** e conferir a integridade

Os dois arquivos Parquet já foram copiados para `Projeto_Mixers_Ethereum/data/processed/`. O Colab os obtém **pelos IDs das cópias no seu Drive**, independentemente de onde a pasta aparece na interface. Conferimos seus SHA-256 com o inventário auditado.

Os downloads são temporários. **Não altera nem sobrescreve os datasets originais.**

In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timezone

# Arquivos COPIADOS para seu próprio projeto, não arquivos da pasta do Bruno.
DADOS_DRIVE = {
    'treino': '1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S',
    'teste2020': '1g22eCccxC4v_S2RmxPhFZyzfF1ARuzPk',
}
INPUTS = {
    'treino': BASE / 'data' / 'processed' / 'features_treino.parquet',
    'teste2020': BASE / 'data' / 'processed' / 'features_teste_20201029.parquet',
}
HASHES_ESPERADOS = {
    'treino': '0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32',
    'teste2020': '6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51',
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

for chave, file_id in DADOS_DRIVE.items():
    arquivo = obter_arquivo_drive(file_id, INPUTS[chave])
    obtido = sha256(arquivo)
    print(chave, arquivo.name, 'SHA-256:', obtido)
    if obtido != HASHES_ESPERADOS[chave]:
        raise ValueError(f'Arquivo {chave} divergente. Não continuar sem auditoria.')
print('Dados do seu Drive: integridade SHA-256 verificada.')

## 4 — Inspecionar as amostras

**Treino (legado):** 123 janelas, 23 positivas e 100 negativas, com positivos de um único contrato. A validação cruzada estratificada por linha não comprova generalização entre contratos.

**Teste (29/10/2020):** apenas contratos com pelo menos 10 transações entram na classificação. Os demais endereços não são negativos verificados.

In [ ]:
treino=pd.read_parquet(INPUTS["treino"])
teste=pd.read_parquet(INPUTS["teste2020"])
MIN_TX=10
SEED=42
assert set(FEATURES).issubset(treino.columns) and set(FEATURES).issubset(teste.columns)
assert {"label","n_tx"}.issubset(treino.columns)
assert {"recipient","n_tx","tornado"}.issubset(teste.columns)
assert set(treino.label.dropna().unique())=={0,1}
treino_e=treino.loc[treino.n_tx>=MIN_TX].copy()
teste_e=teste.loc[teste.n_tx>=MIN_TX].copy()
print("Treino completo:",len(treino),"classes:",treino.label.value_counts().to_dict())
print("Treino elegível:",len(treino_e),"classes:",treino_e.label.value_counts().to_dict())
print("Teste 2020:",len(teste),"janelas; elegíveis:",len(teste_e))
print("Pools de referência:",int(teste_e.tornado.notna().sum()))
display(treino_e.groupby("label")[list(FEATURES)].median().round(3))

## 5 — Validar e treinar o classificador

Random Forest com 100 árvores, profundidade máxima 5, pesos balanceados e semente 42. Oversampling apenas dentro dos folds de treino. As métricas são **internas e exploratórias**, não evidência de teste independente.

In [ ]:
cv=validar_cv(treino_e,seed=SEED,n_splits=5)
display(cv.round(3))
display(cv[["precision","recall","f1"]].agg(["mean","std"]).round(3))
Xt,yt=reamostrar(treino_e[list(FEATURES)],treino_e.label.astype(int),SEED)
modelo=novo_rf(SEED).fit(Xt,yt)
print("RF treinada",modelo.n_estimators,"árvores; max_depth",modelo.max_depth)

## 6 — Aplicar ao dia histórico de 2020

Testamos limiares 0,50 e 0,85 **apenas para caracterização exploratória**. Os demais alertas são endereços sem rótulos confirmados; não podem ser chamados automaticamente de falsos positivos.

A referência de 63 endereços no artigo é um ponto de comparação, **não** um alvo de otimização.

In [ ]:
prob=modelo.predict_proba(teste_e[list(FEATURES)])[:,1]
rows=[]
for limiar in [0.50,0.85]:
    mask=prob>limiar
    recuperados=int(teste_e.loc[mask,"tornado"].notna().sum())
    rows.append({"limiar":limiar,"enderecos_sinalizados":int(mask.sum()),"pools_referencia_recuperados":recuperados,"outros_sinalizados_sem_rotulo_confirmado":int(mask.sum())-recuperados})
resumo=pd.DataFrame(rows)
display(resumo)
print("Confrontar estes números com os relatórios originais antes de afirmar paridade")

## 7 — Registrar a execução **no seu Drive**, sem sobrescrever o legado

Os relatórios são primeiro preparados em `/content/ethmix_reproducao/results/`, depois **enviados por API do Google Drive** para a pasta `Projeto_Mixers_Ethereum/results/`, usando seu ID. Isso funciona mesmo que o projeto não esteja montado em `MyDrive`.

Cada execução cria uma pasta exclusiva em `results/reproducao_consolidada/AAAA...Z/`, com CSVs e manifesto. Os modelos e dados originais não são sobrescritos.

**Atenção:** os resultados ainda precisam ser confrontados com o legado; não se trata de validação cega.

In [ ]:
ts = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = BASE / 'results' / 'reproducao_consolidada' / ts
OUT.mkdir(parents=True, exist_ok=False)
cv.to_csv(OUT / 'cv_folds.csv', index=False)
resumo.to_csv(OUT / 'resumo_2020.csv', index=False)
manifesto = {
    'experimento': 'reproducao_consolidada_01',
    'data_utc': ts,
    'repo': REPO,
    'codigo_ref': CODIGO_REFERENCIA,
    'codigo_sha256': EXPECTED_MODULE_SHA256,
    'arquivos_drive_ids': {'modulos': MODULOS_DRIVE, 'dados': DADOS_DRIVE},
    'dados_sha256': HASHES_ESPERADOS,
    'min_tx': MIN_TX, 'seed': SEED,
    'rf': {'n_estimators': 100, 'max_depth': 5, 'class_weight': 'balanced'},
    'limiares_exploratorios': [0.50, 0.85],
    'ressalvas': ['positivos de um unico contrato',
                 'enderecos nao rotulados nao sao negativos',
                 'teste 2020 previamente explorado']
}
(OUT / 'manifesto.json').write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')

# IDs das pastas que pertencem ao seu Projeto_Mixers_Ethereum.
PASTA_RESULTADOS_ID = '1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'

def obter_ou_criar_pasta(nome, parent_id):
    seguro = nome.replace("'", "\'")
    query = (f"name = '{seguro}' and '{parent_id}' in parents and "
             "mimeType = 'application/vnd.google-apps.folder' and trashed = false")
    encontrados = drive_api.files().list(
        q=query, fields='files(id,name),nextPageToken', supportsAllDrives=True,
        includeItemsFromAllDrives=True, pageSize=100).execute().get('files', [])
    if encontrados:
        return encontrados[0]['id']
    nova = drive_api.files().create(body={
        'name': nome, 'mimeType': 'application/vnd.google-apps.folder',
        'parents': [parent_id]}, fields='id').execute()
    return nova['id']

pasta_experimento = obter_ou_criar_pasta('reproducao_consolidada', PASTA_RESULTADOS_ID)
pasta_execucao = obter_ou_criar_pasta(ts, pasta_experimento)
for nome in ('cv_folds.csv', 'resumo_2020.csv', 'manifesto.json'):
    arquivo = OUT / nome
    tipo = 'application/json' if nome.endswith('.json') else 'text/csv'
    media = MediaFileUpload(str(arquivo), mimetype=tipo, resumable=True)
    drive_api.files().create(body={'name': nome, 'parents': [pasta_execucao]},
                             media_body=media, fields='id').execute()
print('Resultados registrados no seu Drive:')
print('https://drive.google.com/drive/folders/' + pasta_execucao)

## 8 — Por que o artigo sinalizou 63 endereços e nossa reprodução 9?

**Pergunta da pesquisa:** ambos identificaram os três pools Tornado Cash em 29/10/2020, mas o artigo relata **63 endereços sinalizados** e nosso teste, no limiar **0,50**, relata **9**. Vamos investigar sem alterar o modelo nem tentar forçar o resultado a 63.

**O que faremos:** contar as janelas e endereços elegíveis; examinar a distribuição das probabilidades de *mixer*; mostrar os primeiros endereços por pontuação e o comportamento dos alertas ao variar **apenas para diagnóstico** o limiar. Vamos salvar os dados que faltavam no experimento 01: **identidades e scores de todos os endereços elegíveis**.

**O que os gráficos NÃO demonstram:** não indicam qual escolha metodológica dos autores causou a diferença. Não conhecemos a lista dos 63 alertas deles e não devemos confundir alteração do limiar com reprodução fiel do artigo.

**Pré-requisito:** execute as etapas 2–6 deste notebook na mesma sessão, para deixar `teste_e`, `modelo`, `FEATURES` e `prob` disponíveis.


In [ ]:
#@title 5. Investigar população, probabilidades e limites — sem retreinar
import matplotlib.pyplot as plt
from IPython.display import display

predicoes = teste_e[['recipient', 'n_tx', 'tornado']].copy().reset_index(drop=True)
predicoes['prob_mixer'] = modelo.predict_proba(teste_e[list(FEATURES)])[:, 1]
predicoes = predicoes.sort_values('prob_mixer', ascending=False).reset_index(drop=True)
predicoes['sinalizado_050'] = predicoes['prob_mixer'] > 0.50

contagens = pd.DataFrame([{
    'janelas_processadas': len(teste),
    'janelas_elegiveis_n_tx_10': len(teste_e),
    'enderecos_distintos_elegiveis': teste_e['recipient'].nunique(),
    'alertas_no_limiar_050': int(predicoes.sinalizado_050.sum()),
    'pools_tornado_conhecidos_recuperados': int(predicoes.loc[predicoes.sinalizado_050, 'tornado'].notna().sum()),
    'alertas_publicados_pelos_autores': 63,
}])
display(contagens.T.rename(columns={0: 'Quantidade'}))
display(predicoes.head(15).rename(columns={'prob_mixer':'P(mixer)'}))

limiares = np.arange(0.05, 1.00, 0.05)
curva = pd.DataFrame({'limiar': limiares,
                      'enderecos_sinalizados': [int((predicoes.prob_mixer > t).sum()) for t in limiares],
                      'tornado_recuperados': [int(predicoes.loc[predicoes.prob_mixer > t, 'tornado'].notna().sum()) for t in limiares]})

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
axs[0].hist(predicoes.prob_mixer, bins=np.linspace(0, 1, 21))
axs[0].axvline(.5, linestyle='--', label='Limiar original 0,50')
axs[0].set(xlabel='Probabilidade estimada de mixer', ylabel='Janelas elegíveis',
           title='A · Distribuição dos scores')
axs[0].legend(fontsize=8)
axs[1].plot(curva.limiar, curva.enderecos_sinalizados, marker='o', markersize=3,
            label='Alertas na nossa reprodução')
axs[1].axhline(63, linestyle=':', label='63 alertas relatados no artigo')
axs[1].axvline(.5, linestyle='--', label='Limiar original 0,50')
axs[1].set(xlabel='Limiar exploratório (não ajustado)', ylabel='Quantidade de alertas',
           title='B · Sensibilidade ao limiar')
axs[1].legend(fontsize=8)
fig.tight_layout()
plt.show()
print('A curva ajuda a entender o score. Ajustar o limiar para chegar a 63 NÃO é reprodução fiel.')


### Como interpretar esses resultados?

- **Tabela de elegibilidade:** mostra quantas unidades de análise chegam à Random Forest. Se a população elegível não for a mesma dos autores, o total de alertas pode divergir mesmo com modelo equivalente.
- **Gráfico A:** apresenta a distribuição dos *scores* (probabilidades estimadas pelo modelo). Concentração próxima de 0 indica poucos candidatos classificados como mixers no limiar 0,50.
- **Gráfico B:** mostra como mudar o limiar **mudaria** a quantidade de alertas. A linha de 63 é uma **referência do artigo, não uma meta de ajuste**.
- **Ranking:** permite auditar o endereço e a pontuação exata de cada candidato no nosso modelo. Sem a lista dos 63 autores, ainda não é possível conferir coincidência endereço a endereço com o artigo.

**Próxima hipótese científica:** conferir se a população, a transformação dos dados, a seleção dos 100 negativos e as regras de treino/inferência coincidem com o artigo. O e-mail aos autores sobre código, amostras e lista de 63 alertas já foi enviado.


In [ ]:
#@title 6. Salvar evidências do diagnóstico no seu Drive próprio
SALVAR_DIAGNOSTICO = True  #@param {type:"boolean"}
if SALVAR_DIAGNOSTICO:
    from googleapiclient.http import MediaFileUpload
    pasta_local = BASE / 'results' / 'diagnostico_63_vs_9'
    pasta_local.mkdir(parents=True, exist_ok=True)
    predicoes.to_csv(pasta_local / 'scores_elegiveis_2020.csv', index=False)
    curva.to_csv(pasta_local / 'curva_limiares.csv', index=False)
    contagens.to_csv(pasta_local / 'contagens.csv', index=False)
    fig.savefig(pasta_local / 'graficos_diagnostico.png', dpi=160, bbox_inches='tight')
    manifesto_diag = {
        'experimento': 'diagnostico_63_vs_9',
        'executado_em_utc': datetime.now(timezone.utc).isoformat(),
        'limiar_protocolo': 0.50, 'autores_alertas_reportados': 63,
        'treinamento': 'mesmo modelo ajustado na etapa 5 deste notebook',
        'min_tx': MIN_TX, 'seed': SEED,
        'hashes_dados': HASHES_ESPERADOS, 'hashes_codigo': EXPECTED_MODULE_SHA256,
        'observacao': 'diagnostico exploratorio, sem ajuste de hiperparametros para aproximar os 63 alertas',
    }
    (pasta_local / 'manifesto.json').write_text(json.dumps(manifesto_diag, indent=2, ensure_ascii=False))
    nome = 'diagnostico_63_vs_9_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta_id = drive_api.files().create(body={
        'name': nome, 'mimeType': 'application/vnd.google-apps.folder',
        'parents': [PASTA_RESULTADOS_ID]}, fields='id').execute()['id']
    tipos = {'.csv': 'text/csv', '.json': 'application/json', '.png': 'image/png'}
    for arquivo in pasta_local.iterdir():
        if arquivo.suffix not in tipos:
            continue
        drive_api.files().create(body={'name': arquivo.name, 'parents': [pasta_id]},
             media_body=MediaFileUpload(str(arquivo), mimetype=tipos[arquivo.suffix]),
             fields='id').execute()
    print('Diagnóstico salvo:', 'https://drive.google.com/drive/folders/' + pasta_id)
else:
    print('Gráficos visualizados. Salvamento desativado.')


## 9 — Identificar três alertas ainda desconhecidos (consulta externa)

**Pergunta:** os três endereços não rotulados restantes correspondem a contratos conhecidos, contas comuns ou serviços que se parecem estatisticamente com mixers?

**Por quê:** um score alto **não prova** que um contrato é mixer. Primeiro precisamos coletar metadados do endereço e evidências verificáveis; depois examinaremos o comportamento **especificamente em 29/10/2020**. A consulta abaixo usa a API pública do [Blockscout](https://eth.blockscout.com/), em modo somente leitura; não exige token. **Pode haver bloqueio/rate limit**, e ausência de dados não é prova de que o contrato não existe.

**O que será observado:** se a API conseguiu responder, se o endereço é marcado como contrato, se há nome ou código verificado, e referências para inspeção. **Não transforma automaticamente metadados em rótulos negativos ou positivos.** Execute depois de entrar na sua conta Google na etapa 2.

In [ ]:
#@title 7. Consultar metadados públicos e guardar evidências (sem alterar rótulos)
import requests
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload

ALVOS = {
    '0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9': 1.00,
    '0x98f1858c9740b9c5f2cd8abea67948873d38dc39': 0.98,
    '0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18': 0.76,
}

def consultar(url):
    try:
        resposta = requests.get(url, timeout=20, headers={'Accept':'application/json'})
        dados = resposta.json() if resposta.ok else {}
        return resposta.status_code, dados if isinstance(dados, dict) else {}, ''
    except (requests.RequestException, ValueError) as erro:
        return None, {}, type(erro).__name__ + ': ' + str(erro)[:160]

registros, respostas = [], {}
for endereco, score in ALVOS.items():
    base_url = f'https://eth.blockscout.com/api/v2/addresses/{endereco}'
    st, dados, erro = consultar(base_url)
    st_contrato, dados_contrato, erro_contrato = consultar(
        f'https://eth.blockscout.com/api/v2/smart-contracts/{endereco}')
    respostas[endereco] = {'url':base_url, 'http_endereco':st, 'dados_endereco':dados,
                            'http_contrato':st_contrato, 'dados_contrato':dados_contrato,
                            'erro':erro, 'erro_contrato':erro_contrato}
    registros.append({'endereco':endereco, 'score_nosso':score, 'http_endereco':st,
                      'is_contract':dados.get('is_contract'), 'nome':dados.get('name'),
                      'is_verified':dados.get('is_verified'),
                      'http_codigo':st_contrato, 'nome_codigo':dados_contrato.get('name'),
                      'fonte':base_url, 'erro':erro or erro_contrato})

metadados = pd.DataFrame(registros)
display(metadados)
print('Metadados são pistas, NÃO rótulos confirmados. Inspecionar atividade histórica de 2020 depois.')

SALVAR_CONSULTA = True  #@param {type:"boolean"}
if SALVAR_CONSULTA:
    pasta_local = BASE / 'results' / 'triagem_onchain_3'
    pasta_local.mkdir(parents=True, exist_ok=True)
    metadados.to_csv(pasta_local/'metadados_3_enderecos.csv', index=False)
    (pasta_local/'respostas_publicas.json').write_text(
        json.dumps({'data_utc':datetime.now(timezone.utc).isoformat(), 'respostas':respostas},
                   ensure_ascii=False, indent=2, default=str))
    nome = 'triagem_onchain_3_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    pasta_id = drive_api.files().create(body={
        'name':nome, 'mimeType':'application/vnd.google-apps.folder',
        'parents':[PASTA_RESULTADOS_ID]}, fields='id').execute()['id']
    for arquivo in (pasta_local/'metadados_3_enderecos.csv', pasta_local/'respostas_publicas.json'):
        media = MediaFileUpload(str(arquivo), mimetype='text/csv' if arquivo.suffix=='.csv' else 'application/json')
        drive_api.files().create(body={'name':arquivo.name, 'parents':[pasta_id]},
                                 media_body=media, fields='id').execute()
    print('Evidências salvas:', 'https://drive.google.com/drive/folders/' + pasta_id)


## 10 — Auditoria histórica: o que os três contratos fizeram em 29/10/2020?

**Pergunta:** as pontuações de 1,00, 0,98 e 0,76 podem ser conferidas a partir das transações reais daquele dia?

**Por que essa etapa é melhor do que consultar apenas o nome atual:** o nome no explorador não revela a atividade de 2020. Aqui usamos a **cópia do arquivo intermediário de transações de 29/10/2020** no seu Drive, filtramos apenas os três destinatários e recalculamos as quatro features com a função versionada do projeto. É uma **auditoria de consistência com os dados intermediários**, não uma confirmação independente das transações on-chain nem da identidade econômica dos protocolos.

**O que observar:** tabela de contagens; diferenças absolutas entre features recalculadas e processadas; gráfico que explica as quatro medidas. Para os três endereços, verifique que `n_tx` é respectivamente **53, 10 e 10** (conforme os relatórios anteriores). **Se qualquer comparação falhar, não interprete como conclusão: investigue esquema/fonte/transformação.**

**Observação sobre o proxy `0x5e60…`:** o JSON anterior aponta implementação inicial `0x68b37D89e1AEd3Dd348AD8B1cFC1E678539431D7` e implementação atual `0xee2fCEcFb2075A85a31eE8676cE9Ae5f3c3be6D7`. Não sabemos **a data da troca**; isso exige análise de eventos `Upgraded` e histórico de execução no bloco, fora do escopo deste check de features.

**Ação:** basta executar a próxima célula depois da etapa 2 (login no Google e módulos carregados). Ela produz os gráficos e pode gravar um novo relatório sem modificar os dados. O download intermediário tem cerca de 141 MB.

In [ ]:
#@title 8. Auditar transações de 2020 e comparar as 4 features
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload
import json, hashlib

SALVAR_AUDITORIA_HISTORICA = True  #@param {type:"boolean"}
ARQUIVO_INTERIM_2020 = '1Ei7sCgYdFfX-AjrK6uazOvbrf28d1Px7'
alvos = ['0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9',
         '0x98f1858c9740b9c5f2cd8abea67948873d38dc39',
         '0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18']
origem = obter_arquivo_drive(ARQUIVO_INTERIM_2020, BASE/'data'/'interim'/'test_20201029.parquet')
transacoes = pd.read_parquet(origem)
necessarias = {'recipient','hash','time','sender','call_count'}
assert necessarias.issubset(transacoes.columns), f'Colunas esperadas ausentes: {necessarias-set(transacoes.columns)}'
filtradas = transacoes.loc[transacoes['recipient'].astype(str).str.lower().isin(alvos)].copy()
assert not filtradas.empty, 'Os três destinatários não aparecem nesse arquivo intermediário.'
filtradas = preparar(filtradas)
assert filtradas['dia'].astype(str).eq('2020-10-29').all(), 'Há transações fora do dia de análise.'
recalculado = calcular_features(filtradas).drop(columns=['dia']).set_index('recipient')
ref = pd.read_parquet(INPUTS['teste2020']).set_index('recipient').loc[alvos]
comparacao = recalculado[['n_tx',*FEATURES]].join(ref[['n_tx',*FEATURES]], lsuffix='_novo', rsuffix='_referencia')
for metrica in ('n_tx',*FEATURES):
    comparacao[f'delta_{metrica}'] = (comparacao[f'{metrica}_novo']-comparacao[f'{metrica}_referencia']).abs()
comparacao['paridade_numerica'] = comparacao[[f'delta_{x}' for x in ('n_tx',*FEATURES)]].max(axis=1).le(1e-9)
resumo_tx = filtradas.groupby('recipient',as_index=False).agg(n_tx=('hash','size'),
                         remetentes_distintos=('sender','nunique'),
                         chamadas_medias=('call_count','mean'))
display(comparacao.round(6))
display(resumo_tx.round(2))
assert sorted(resumo_tx.n_tx.tolist()) == [10,10,53], 'Contagens inesperadas: investigar dados antes de concluir.'
fig_h, ax = plt.subplots(figsize=(9, 3.8))
view = comparacao[[f'delta_{f}' for f in FEATURES]].T
ax.imshow(np.log10(view.to_numpy()+1e-12), aspect='auto', vmin=-12, vmax=0, cmap='Blues')
ax.set_yticks(range(len(FEATURES)), list(FEATURES))
ax.set_xticks(range(len(alvos)), [x[:8]+'…'+x[-4:] for x in comparacao.index])
ax.set_title('Diferença absoluta das features recalculadas (quanto mais claro, menor)')
for i in range(len(FEATURES)):
    for j in range(len(alvos)):
        ax.text(j,i,f'{view.iloc[i,j]:.1e}',ha='center',va='center',fontsize=8)
fig_h.tight_layout(); plt.show()
print('Paridade numérica em todos os três:', bool(comparacao.paridade_numerica.all()))
print('AVISO: nomes atuais de proxies NÃO estabelecem sua função ou implementação em 2020.')
if SALVAR_AUDITORIA_HISTORICA:
    out=BASE/'results'/'auditoria_historica_2020'; out.mkdir(parents=True,exist_ok=True)
    comparacao.reset_index().to_csv(out/'comparacao_features_3.csv',index=False)
    resumo_tx.to_csv(out/'resumo_transacoes_3.csv',index=False)
    fig_h.savefig(out/'delta_features_3.png',dpi=150,bbox_inches='tight')
    (out/'manifesto.json').write_text(json.dumps({'data_utc':datetime.now(timezone.utc).isoformat(),
       'dataset_drive_id':ARQUIVO_INTERIM_2020,'sha256_interim':hashlib.sha256(origem.read_bytes()).hexdigest(),
       'origem':'interim 2020; auditoria de consistência, não auditoria independente da blockchain',
       'contratos':alvos,'paridade_por_contrato':comparacao['paridade_numerica'].to_dict()},indent=2,ensure_ascii=False))
    pasta=drive_api.files().create(body={'name':'auditoria_historica_2020_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'),
       'mimeType':'application/vnd.google-apps.folder','parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id').execute()['id']
    for arq in out.iterdir():
        if arq.suffix not in ('.csv','.png','.json'): continue
        mime={'.csv':'text/csv','.png':'image/png','.json':'application/json'}[arq.suffix]
        drive_api.files().create(body={'name':arq.name,'parents':[pasta]},media_body=MediaFileUpload(str(arq),mimetype=mime),fields='id').execute()
    print('Auditoria histórica salva:', 'https://drive.google.com/drive/folders/'+pasta)


## 11 — Verificação externa de transações e upgrades do proxy (2020)

**Objetivo:** conferir a atividade dos três contratos e os eventos `Upgraded` do proxy em fonte pública independente do arquivo Blockchair. A API Blockscout é paginada; os registros contemporâneos de 2026 **não provam**, isoladamente, o estado do contrato em 2020.

**Método:** paginar separadamente transações dos três endereços e logs do proxy, observar timestamps de 29/10/2020 (UTC), extrair eventos `Upgraded(address)` e produzir tabelas com a **cobertura real de consulta**. Cada trilha registra `completa_ate_data`, `paginas` e eventuais erros. Comparações numéricas com o parquet só são conclusivas se o percurso alcançar todo o período de interesse, sem interrupções.

**Não assume** que `Upgraded` é o único mecanismo que pode afetar o comportamento, nem que os nomes atuais dos contratos descrevem sua atividade em 2020. Não modifica rótulos e não exige credenciais Ethereum. Mantenha o limite de páginas para evitar consultas excessivas; se o período não for alcançado, a análise será explicitamente **inconclusiva**.

**Pré-requisito:** execute a etapa 2 para autenticar Google Drive e definir `drive_api`, `BASE` e `obter_arquivo_drive`.

In [34]:
#@title 9. Consultar transações/eventos on-chain com limites e registrar a cobertura
import requests, pandas as pd, json, time
from datetime import datetime, timezone
from pathlib import Path
from googleapiclient.http import MediaFileUpload

ALVOS_HIST = [
    '0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9',
    '0x98f1858c9740b9c5f2cd8abea67948873d38dc39',
    '0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18',
]
PROXY = ALVOS_HIST[0]
TOPIC_UPGRADED = '0xbc7cd75a20ee27fd9adebab32041f755214dbc6bffa90cc0225b39da2e5c2d3b'
DIA_INICIO, DIA_FIM = '2020-10-29', '2020-10-30'
MAX_PAGINAS = 30  #@param {type:"integer"}
SALVAR_EVENTOS = True  #@param {type:"boolean"}

# O Blockscout retorna páginas da mais recente para a mais antiga.
# Não concluir ausência de evento se o histórico não alcançar o período analisado.
def historico(endereco, tipo, max_paginas=MAX_PAGINAS):
    endpoint = f'https://eth.blockscout.com/api/v2/addresses/{endereco}/{tipo}'
    sessao = requests.Session()
    sessao.headers.update({'Accept': 'application/json', 'User-Agent':'Pesquisa academica de reprodutibilidade'})
    cursor, vistos, registros = None, set(), []
    situacao = {'endereco':endereco,'tipo':tipo,'paginas':0,'http_ultimo':None,
                'completa_ate_data':False,'fim_da_paginacao':False,'erro':'','mais_antigo':''}
    for pagina in range(max_paginas):
        try:
            resp = sessao.get(endpoint, params=cursor or {}, timeout=25)
            situacao['http_ultimo'] = resp.status_code
            resp.raise_for_status()
            dados = resp.json()
            if not isinstance(dados, dict) or not isinstance(dados.get('items'),list):
                raise ValueError('Formato inesperado da resposta')
            itens = dados['items']
            situacao['paginas'] += 1
            registros.extend(itens)
            datas = [str(x.get('timestamp') or x.get('block_timestamp') or '')[:10] for x in itens]
            datas = [d for d in datas if len(d)==10 and d[:4].isdigit()]
            if datas:
                menor = min(datas)
                situacao['mais_antigo'] = menor if not situacao['mais_antigo'] else min(menor,situacao['mais_antigo'])
                if menor < DIA_INICIO:
                    situacao['completa_ate_data'] = True
                    break
            novo_cursor = dados.get('next_page_params')
            if not novo_cursor:
                situacao['fim_da_paginacao'] = True
                situacao['completa_ate_data'] = True
                break
            assinatura = json.dumps(novo_cursor,sort_keys=True)
            if assinatura in vistos:
                situacao['erro'] = 'Cursor repetido; paginação interrompida'
                break
            vistos.add(assinatura)
            cursor = novo_cursor
            time.sleep(0.15)
        except (requests.RequestException,ValueError,TypeError) as e:
            situacao['erro'] = type(e).__name__ + ': ' + str(e)[:200]
            break
    return registros, situacao

situacoes, tx_dia, upgrades = [], [], []
for endereco in ALVOS_HIST:
    itens, cobertura = historico(endereco,'transactions')
    situacoes.append(cobertura)
    for it in itens:
        data = str(it.get('timestamp') or it.get('block_timestamp') or '')
        if DIA_INICIO <= data[:10] < DIA_FIM:
            tx_dia.append({'endereco':endereco,'hash':it.get('hash'), 'timestamp':data,
                           'bloco':it.get('block_number'), 'de':(it.get('from') or {}).get('hash'),
                           'para':(it.get('to') or {}).get('hash'), 'status':it.get('status'),
                           'metodo':it.get('method')})

itens_logs, cob_logs = historico(PROXY,'logs')
situacoes.append(cob_logs)
for log in itens_logs:
    topicos = log.get('topics') or []
    topic0 = topicos[0] if topicos else ''
    if isinstance(topic0,dict): topic0 = topic0.get('hex') or topic0.get('value') or ''
    if str(topic0).lower() == TOPIC_UPGRADED:
        data = str(log.get('timestamp') or log.get('block_timestamp') or '')
        implementacao = ''
        if len(topicos)>1 and isinstance(topicos[1],str):
            implementacao = '0x'+topicos[1][-40:]
        upgrades.append({'data':data,'bloco':log.get('block_number'),
                         'tx_hash':log.get('transaction_hash'),'implementacao':implementacao})

coberturas = pd.DataFrame(situacoes)
transacoes_externas = pd.DataFrame(tx_dia,columns=['endereco','hash','timestamp','bloco','de','para','status','metodo'])
eventos_upgrade = pd.DataFrame(upgrades,columns=['data','bloco','tx_hash','implementacao'])
display(coberturas)
display(transacoes_externas.groupby('endereco').size().rename('tx_encontradas_na_API').to_frame() if not transacoes_externas.empty else pd.DataFrame({'observacao':['Nenhuma tx da data encontrada nos trechos consultados']}))
display(eventos_upgrade)
for _,r in coberturas.iterrows():
    if not r['completa_ate_data']:
        print('INCONCLUSIVO:',r['tipo'],r['endereco'][:12],'- paginação não alcançou todo o período; não inferir ausência de eventos')
if SALVAR_EVENTOS:
    pasta_local=BASE/'results'/'verificacao_externa_2020'
    pasta_local.mkdir(parents=True,exist_ok=True)
    coberturas.to_csv(pasta_local/'cobertura_api.csv',index=False)
    transacoes_externas.to_csv(pasta_local/'transacoes_20201029_api.csv',index=False)
    eventos_upgrade.to_csv(pasta_local/'upgrades_proxy_api.csv',index=False)
    manifesto={'experimento':'verificacao_externa_2020','coletado_em_utc':datetime.now(timezone.utc).isoformat(),
              'fonte':'https://eth.blockscout.com/api/v2','limite_paginas_por_consulta':MAX_PAGINAS,
              'nota':'Paginação incompleta impede ausência ou contagens conclusivas; não modifica ground truth',
              'cobertura':situacoes}
    (pasta_local/'manifesto.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')
    pasta=drive_api.files().create(body={'name':'verificacao_externa_2020_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'),
        'mimeType':'application/vnd.google-apps.folder','parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id').execute()['id']
    for arq in pasta_local.iterdir():
        mime='application/json' if arq.suffix=='.json' else 'text/csv'
        drive_api.files().create(body={'name':arq.name,'parents':[pasta]},
             media_body=MediaFileUpload(str(arq),mimetype=mime),fields='id').execute()
    print('Auditoria externa salva: https://drive.google.com/drive/folders/'+pasta)


,endereco,tipo,paginas,http_ultimo,completa_ate_data,fim_da_paginacao,erro,mais_antigo
0,0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9,transactions,30,200,False,False,,2020-11-06
1,0x98f1858c9740b9c5f2cd8abea67948873d38dc39,transactions,16,200,True,False,,2020-10-27
2,0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18,transactions,3,200,True,False,,2020-10-26
3,0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9,logs,30,200,False,False,,2020-11-05


,tx_encontradas_na_API
endereco,
0x98f1858c9740b9c5f2cd8abea67948873d38dc39,10
0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18,10


,data,bloco,tx_hash,implementacao
0,2022-04-25T15:12:03.000000Z,14654508,,0xee2fcecfb2075a85a31ee8676ce9ae5f3c3be6d7
1,2020-11-12T11:43:27.000000Z,11242610,,0x7497fd42b322d734c4877a35651971206eadced6


INCONCLUSIVO: transactions 0x5e60fcca90 - paginação não alcançou todo o período; não inferir ausência de eventos
INCONCLUSIVO: logs 0x5e60fcca90 - paginação não alcançou todo o período; não inferir ausência de eventos
Auditoria externa salva: https://drive.google.com/drive/folders/1Xz6tBlOqQcMzzoX1uGycbZPwsyKv17iO


## 12 — Estado histórico do proxy no fechamento de 29/10/2020

**Pergunta:** qual era o endereço de implementação armazenado no slot EIP-1967 de `0x5e60...27aa9` no último bloco do experimento de 2020?

**Amostra de dados:** o dump bruto Blockchair de 29/10/2020 contém **1.114.702** transações nos blocos **11.148.274 a 11.154.799**. Já validamos, em auditoria externa, os **20/20 hashes** dos contratos SmartEther e DSProxy (também remetentes, destinatários e blocos): [EXP-01G no GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection/blob/main/docs/experiments/EXP-01G-auditoria-hashes-blockchair-blockscout.md).

**Método:** consultar `eth_getStorageAt` (somente leitura) no *slot* de implementação EIP-1967, **no bloco 11.154.799**. A chamada exige que o endpoint Ethereum possua **estado de arquivo (*archive*)**. O notebook tenta dois provedores públicos sem chave, guarda respostas e erros e **não substitui um resultado desconhecido por um endereço suposto**. Opcionalmente, uma busca limitada a esse dia procura eventos `Upgraded`; falhas, limitações ou ausência de logs não devem ser tratadas como confirmação automática de que não houve upgrade.

**Limite científico:** valor de storage no último bloco demonstra a implementação **no fechamento do dia**, não em todas as 53 transações distribuídas ao longo do dia. Reconstituir mudanças intradiárias exigirá logs históricos completos ou consultas nos blocos de cada transação.

Execute a célula após a etapa 2 (login Google). Se os endpoints públicos não permitirem acesso ao estado histórico, registraremos a limitação e a opção de provedor de arquivo com autorização, sem inventar resultados.


In [ ]:
#@title 10. Consultar implementação EIP-1967 no bloco de 29/10/2020
import requests, json, re, pandas as pd
from datetime import datetime, timezone
from googleapiclient.http import MediaFileUpload

SALVAR_ESTADO_PROXY = True  #@param {type:"boolean"}
ENDERECO_PROXY = '0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9'
BLOCO_FINAL_20201029 = 11154799  # bloco máximo observado no dump Blockchair 29/10/2020
SLOT_EIP1967_IMPLEMENTACAO = '0x360894a13ba1a3210667c828492db98dca3e2076cc3735a920a3ca505d382bbc'
ENDPOINTS_RPC = ['https://ethereum-rpc.publicnode.com', 'https://eth.llamarpc.com']

leituras = []
for url in ENDPOINTS_RPC:
    registro = {'provedor':url, 'bloco':BLOCO_FINAL_20201029, 'http':None,
                'valor_slot':'', 'implementacao_no_fim_do_dia':'', 'erro':''}
    try:
        payload = {'jsonrpc':'2.0', 'method':'eth_getStorageAt',
                   'params':[ENDERECO_PROXY, SLOT_EIP1967_IMPLEMENTACAO, hex(BLOCO_FINAL_20201029)], 'id':1}
        resp = requests.post(url, json=payload, timeout=20)
        registro['http'] = resp.status_code
        resp.raise_for_status()
        dados = resp.json()
        valor = dados.get('result') if isinstance(dados,dict) else None
        if isinstance(valor,str) and re.fullmatch('0x[0-9a-fA-F]{64}',valor):
            registro['valor_slot'] = valor.lower()
            impl = '0x'+valor.lower()[-40:]
            if int(impl,16)!=0:
                registro['implementacao_no_fim_do_dia'] = impl
            else:
                registro['erro'] = 'Slot zero: não identifica implementação histórica'
        else:
            registro['erro'] = str(dados.get('error') if isinstance(dados,dict) else dados)[:350]
    except (requests.RequestException,ValueError) as exc:
        registro['erro'] = type(exc).__name__ + ': ' + str(exc)[:250]
    leituras.append(registro)

estados = pd.DataFrame(leituras)
display(estados)
validos = estados.loc[estados.implementacao_no_fim_do_dia.ne(''), 'implementacao_no_fim_do_dia'].unique().tolist()
status = ('verificado_no_fim_do_dia' if len(validos)==1 else
          'divergencia_entre_provedores' if len(validos)>1 else 'inconclusivo_sem_archive')
print('Resultado da leitura do slot:', status)

# Busca concentrada no dia do experimento; não substitui histórico completo de upgrades.
consulta_logs = {'endpoint':'https://eth.blockscout.com/api', 'http':None,
                 'status':'', 'erro':'', 'eventos_no_dia':[], 'resposta_resumida':''}
try:
    params = {'module':'logs', 'action':'getLogs', 'fromBlock':11148274,
              'toBlock':BLOCO_FINAL_20201029, 'address':ENDERECO_PROXY,
              'topic0':'0xbc7cd75a20ee27fd9adebab32041f755214dbc6bffa90cc0225b39da2e5c2d3b'}
    r = requests.get(consulta_logs['endpoint'],params=params,timeout=25)
    consulta_logs['http'] = r.status_code
    r.raise_for_status()
    dados = r.json()
    if isinstance(dados,dict):
        consulta_logs['status'] = str(dados.get('status',''))
        eventos = dados.get('result')
        if isinstance(eventos,list):
            consulta_logs['eventos_no_dia'] = eventos[:100]
        else:
            consulta_logs['resposta_resumida'] = str(eventos or dados.get('message',''))[:500]
    else: consulta_logs['erro'] = 'Formato inesperado da API logs'
except (requests.RequestException,ValueError) as e:
    consulta_logs['erro'] = type(e).__name__ + ': ' + str(e)[:250]
print('Busca de Upgraded no dia:',len(consulta_logs['eventos_no_dia']),
      'eventos retornados. HTTP:',consulta_logs['http'], 'status:',consulta_logs['status'])
if status!='verificado_no_fim_do_dia':
    print('INCONCLUSIVO: nenhum provedor confirmou o estado histórico. Não presumir implementação inicial.')
if len(validos)>1:
    print('DIVERGÊNCIA de provedores: investigar confiabilidade antes de usar qualquer endereço.')

if SALVAR_ESTADO_PROXY:
    pasta_local = BASE/'results'/'estado_historico_proxy_2020'
    pasta_local.mkdir(parents=True,exist_ok=True)
    estados.to_csv(pasta_local/'leitura_slot_eip1967.csv',index=False)
    (pasta_local/'consulta_logs_dia.json').write_text(
        json.dumps(consulta_logs,ensure_ascii=False,indent=2),encoding='utf-8')
    manifesto = {'data_utc':datetime.now(timezone.utc).isoformat(), 'experimento':'historico_proxy_2020',
                'bloco_final':BLOCO_FINAL_20201029, 'slot':SLOT_EIP1967_IMPLEMENTACAO,
                'status':status, 'implementacoes_respondidas':validos,
                'nota':'Somente o estado no bloco final pode ser inferido de leitura válida; eventos intradiários separados.'}
    (pasta_local/'manifesto.json').write_text(
        json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')
    pasta_id = drive_api.files().create(body={
        'name':'estado_historico_proxy_2020_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'),
        'mimeType':'application/vnd.google-apps.folder','parents':[PASTA_RESULTADOS_ID]},fields='id').execute()['id']
    for arq in pasta_local.iterdir():
        mime = 'text/csv' if arq.suffix=='.csv' else 'application/json'
        drive_api.files().create(body={'name':arq.name,'parents':[pasta_id]},
            media_body=MediaFileUpload(str(arq),mimetype=mime),fields='id').execute()
    print('Evidências históricas salvas: https://drive.google.com/drive/folders/'+pasta_id)


## 13 — Diagnóstico controlado da diferença 63 × 9: treino, balanceamento e elegibilidade

**Objetivo:** investigar quantitativamente **quais decisões nossas alteram o número de alertas**, mantendo o critério original de decisão `P(mixer) > 0,50`. Este é um **estudo de sensibilidade**, NÃO uma tentativa de ajustar o experimento para fabricar 63 alertas.

**Já verificado documentalmente no artigo:** Random Forest (100 árvores, profundidade 5), reamostragem de classes, pesos automáticos, 100 negativos inspecionados e filtro de pelo menos 10 transações para os negativos. O artigo não fornece lista exata dos negativos, discretização das entropias, sementes de aleatoriedade ou todos os detalhes de agregação.

**Nossa construção registrada:** 23 janelas positivas de **um único contrato** Tornado 10 ETH em março/2025 e 100 janelas negativas, provenientes da seleção auditada de endereços em 15/03/2025. O notebook legado 03_v2 documenta a opção `INCLUIR_ROUTER=True` (se os arquivos via Router existirem). Essas opções **não podem ser confirmadas como equivalentes à implementação dos autores**.

A etapa a seguir mantém a população e as quatro features fixas para comparar, separadamente, **reamostragem × pesos automáticos × semente**. Em seguida, mantendo o modelo-base fixo, avalia como o filtro `n_tx` altera o universo elegível. Assim evitamos misturar alterações do treino com alterações da população.

**Pré-requisito:** rodar as etapas 2 a 6 para definir `treino_e`, `teste`, `teste_e`, `modelo`, `FEATURES`, `reamostrar`, `SEED`, `BASE` e `drive_api`. A base e o modelo da etapa 5 são **preservados**.

**Atenção:** validação cruzada por linha compartilha o único contrato positivo entre folds. F1 elevado nessa CV não mede generalização para novos mixers. A análise abaixo **não fornece** novo ground truth nem valida os 60 endereços do artigo.

In [ ]:
#@title 10. Comparar cenários controlados sem alterar o modelo de referência
import pandas as pd, numpy as np, json, hashlib
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from googleapiclient.http import MediaFileUpload

SALVAR_SENSIBILIDADE = True  #@param {type:"boolean"}
LIMIAR_FIXO = 0.50
FEATURES_S13 = list(FEATURES)
assert all(x in treino_e.columns and x in teste.columns for x in FEATURES_S13)
assert {'recipient','n_tx','tornado'}.issubset(teste.columns)
assert {'recipient','dia','label','n_tx'}.issubset(treino_e.columns)

# Concentração dos exemplos por endereço e período: somente diagnóstico, sem alterar os rótulos.
resumo_treino = (treino_e.groupby('label', dropna=False)
                .agg(janelas=('recipient','size'),
                     enderecos_distintos=('recipient','nunique'),
                     dias_distintos=('dia','nunique'),
                     n_tx_mediana=('n_tx','median'))
                .reset_index())
print('Composição do conjunto de treino:')
display(resumo_treino)

Xtr = treino_e[FEATURES_S13]
ytr = treino_e['label'].astype(int)
Xte = teste_e[FEATURES_S13]
baseline_carregado = modelo.predict_proba(Xte)[:, 1]

# Os cenários mudam um fator por vez; o artigo não especifica todas essas alternativas.
CENARIOS = [
    ('base_oversampling_e_pesos_seed42', True, 'balanced', 42),
    ('so_pesos_seed42', False, 'balanced', 42),
    ('so_oversampling_seed42', True, None, 42),
    ('sem_balanceamento_seed42', False, None, 42),
    ('base_seed7', True, 'balanced', 7),
    ('base_seed123', True, 'balanced', 123),
]
resultados, alertas, probabilidades = [], [], {}
for nome, usar_oversampling, peso, semente in CENARIOS:
    xx, yy = (reamostrar(Xtr, ytr, seed=semente) if usar_oversampling else (Xtr, ytr))
    m = RandomForestClassifier(n_estimators=100, max_depth=5,
                               class_weight=peso, random_state=semente)
    m.fit(xx, yy)
    scores = m.predict_proba(Xte)[:, 1]
    probabilidades[nome] = scores
    mask = scores > LIMIAR_FIXO
    ativos = teste_e.loc[mask, ['recipient', 'n_tx', 'tornado']].copy()
    ativos['score'] = scores[mask]
    ativos['cenario'] = nome
    alertas.append(ativos)
    resultados.append(dict(cenario=nome,oversampling=usar_oversampling,
                           class_weight=str(peso),seed=semente,
                           n_train_antes=len(ytr),n_train_depois=len(yy),
                           positivos_depois=int(np.sum(np.asarray(yy)==1)),
                           negativos_depois=int(np.sum(np.asarray(yy)==0)),
                           n_elegiveis=len(teste_e),alertas=int(mask.sum()),
                           pools_artigo_recuperados=int(teste_e.loc[mask,'tornado'].notna().sum())))

comparacao = pd.DataFrame(resultados)
alertas_cenarios = pd.concat(alertas, ignore_index=True)
base = probabilidades[CENARIOS[0][0]]
identico_carregado = bool(np.allclose(base, baseline_carregado, atol=1e-12, rtol=0))
comparacao['sobreposicao_com_base'] = [
    len(set(teste_e.loc[probabilidades[c]>LIMIAR_FIXO,'recipient']) &
        set(teste_e.loc[base>LIMIAR_FIXO,'recipient'])) for c in comparacao.cenario
]
comparacao['igualdade_scores_com_base'] = [bool(np.allclose(probabilidades[c],base,atol=1e-12,rtol=0))
                                          for c in comparacao.cenario]
print('Modelo-base recalculado coincide com o modelo anteriormente treinado:', identico_carregado)
if not identico_carregado:
    print('ATENÇÃO: base diferente da sessão; interpretar apenas como sensibilidade, investigar reprodução.')
print('Cenários:')
display(comparacao)

# Alterar só o universo elegível, utilizando sempre o mesmo modelo de referência da etapa 5.
elegibilidade=[]
for minimo in [5,10,20]:
    eleg = teste.loc[teste.n_tx>=minimo].copy()
    s = modelo.predict_proba(eleg[FEATURES_S13])[:,1]
    elegibilidade.append(dict(min_tx=minimo,janelas_elegiveis=len(eleg),
                                enderecos_unicos=eleg.recipient.nunique(),
                                alertas=int((s>LIMIAR_FIXO).sum()),
                                pools_artigo_recuperados=int(eleg.loc[s>LIMIAR_FIXO,'tornado'].notna().sum())))
comparacao_elegibilidade = pd.DataFrame(elegibilidade)
print('Mesmo RF da etapa 5, mudando apenas mínimo de transações no dia de teste:')
display(comparacao_elegibilidade)

fig_s13, ax = plt.subplots(figsize=(10,4))
ax.barh(comparacao.cenario, comparacao.alertas)
ax.axvline(63, linestyle='--', label='63 relatados pelo artigo (referência, não alvo)')
ax.set(xlabel='Alertas P(mixer) > 0,50', title='Sensibilidade ao protocolo de balanceamento e à semente')
ax.legend(fontsize=8)
fig_s13.tight_layout()
plt.show()
print('IMPORTANTE: diferenças entre cenários NÃO demonstram a causa dos 63 alertas do artigo.')

if SALVAR_SENSIBILIDADE:
    out = BASE / 'results' / 'sensibilidade_63_vs_9_protocolo'
    out.mkdir(parents=True, exist_ok=True)
    arquivos={
       'composicao_treino.csv':resumo_treino,
       'comparacao_cenarios.csv':comparacao,
       'comparacao_elegibilidade.csv':comparacao_elegibilidade,
       'alertas_cenarios.csv':alertas_cenarios,
    }
    for nome, df in arquivos.items(): df.to_csv(out / nome,index=False)
    fig_s13.savefig(out / 'grafico_sensibilidade.png',dpi=150,bbox_inches='tight')
    manifesto13 = dict(experimento='EXP-01I-sensibilidade-de-protocolo',
        executado_utc=datetime.now(timezone.utc).isoformat(),
        limiar_inalterado=LIMIAR_FIXO,min_tx_referencia=10,
        base_igual_modelo_carregado=identico_carregado,
        algoritmos='RF 100 arvores, max_depth=5; somente balanceamento/semente variam',
        hashes_dados=HASHES_ESPERADOS if 'HASHES_ESPERADOS' in globals() else None,
        limites=['Nao altera amostragem dos 100 negativos nem o tratamento Router',
                 'Nao conhece a lista de 63 alertas do artigo',
                 'Positivos de um unico contrato; nao mede generalizacao para contratos novos',
                 'Variacao de n_tx altera populacao de teste, nao prova equivalencia com artigo'])
    (out/'manifesto.json').write_text(json.dumps(manifesto13,ensure_ascii=False,indent=2))
    pasta=drive_api.files().create(body={
        'name':'sensibilidade_63_vs_9_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'),
        'mimeType':'application/vnd.google-apps.folder',
        'parents':['1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI']},fields='id').execute()['id']
    for arq in sorted(out.iterdir()):
        if arq.suffix not in ('.csv','.json','.png'): continue
        mime={'.csv':'text/csv','.json':'application/json','.png':'image/png'}[arq.suffix]
        drive_api.files().create(body={'name':arq.name,'parents':[pasta]},
            media_body=MediaFileUpload(str(arq),mimetype=mime),fields='id').execute()
    print('Análise salva: https://drive.google.com/drive/folders/'+pasta)
